In [1]:
# %% 셀 1: eval.jsonl 파싱 + 채널 통계 (나중에 bucket별 분석용)
import os
import re
import json
import glob
import random
import hashlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
INST_DIR   = "./data/6_telop_instances"

N_BUCKETS  = 5  # 나중에 평가 시 bucket별 분석용
SEED       = 42


# -------------------------
# 1) eval.jsonl에서 전체 채널 목록 추출
# -------------------------
def extract_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(
            part.get("text", "") for part in msg_content if isinstance(part, dict)
        )
    return ""


eval_channels: set[str] = set()
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in tqdm(lines, desc="eval.jsonl 파싱"):
    ex = json.loads(line)
    user_msg = next(m for m in ex["messages"] if m["role"] == "user")
    text = extract_user_text(user_msg["content"])
    m_ch = re.search(r"채널:\s*([^\n]+)", text)
    if m_ch:
        eval_channels.add(m_ch.group(1).strip())

all_channels = sorted(eval_channels)
print(f"\neval.jsonl의 고유 채널 수: {len(all_channels)}")


# -------------------------
# 2) 채널별 통계 (나중에 bucket별 분석용)
# -------------------------
def channel_stats(channel: str) -> dict | None:
    ch_dir = os.path.join(INST_DIR, channel)
    if not os.path.isdir(ch_dir):
        return None

    json_paths = sorted(glob.glob(os.path.join(glob.escape(ch_dir), "*.json")))
    if not json_paths:
        return None

    n_videos    = 0
    n_instances = 0
    dur_sum     = 0.0
    textlen_sum = 0
    density_sum = 0.0

    for jp in json_paths:
        try:
            with open(jp, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        instances = data.get("instances", [])
        n_videos += 1

        if not instances:
            continue

        starts = np.array([inst["start_sec"] for inst in instances], dtype=np.float64)
        ends   = np.array([inst["end_sec"]   for inst in instances], dtype=np.float64)
        texts  = [inst["text"] for inst in instances]

        durs    = np.clip(ends - starts, 0, None)
        vid_len = float(ends.max()) if len(ends) else 0.0

        n_instances += len(instances)
        dur_sum     += float(durs.sum())
        textlen_sum += sum(len(t) for t in texts)

        if vid_len > 0:
            density_sum += float(durs.sum()) / vid_len

    if n_videos == 0:
        return None

    return {
        "n_videos":     n_videos,
        "n_instances":  n_instances,
        "avg_count":    n_instances / n_videos,
        "avg_duration": (dur_sum / n_instances) if n_instances > 0 else 0.0,
        "avg_text_len": (textlen_sum / n_instances) if n_instances > 0 else 0.0,
        "avg_density":  density_sum / n_videos,
    }


stats_rows = []
for ch in tqdm(all_channels, desc="채널 통계 계산"):
    s = channel_stats(ch)
    if s is None:
        continue
    stats_rows.append({"channel": ch, **s})

stats_df = pd.DataFrame(stats_rows)
stats_df = stats_df.sort_values("avg_density").reset_index(drop=True)
stats_df["bucket"] = pd.qcut(
    stats_df["avg_density"],
    q=N_BUCKETS,
    labels=list(range(N_BUCKETS)),
).astype(int)

print(f"\n✅ 전체 채널: {len(stats_df)}개")
print(stats_df.groupby("bucket")["avg_density"].agg(["count", "min", "max", "mean"]))


# -------------------------
# 3) 영상별 swap 채널 미리 결정 (재현성 보장)
# -------------------------
def get_swap_channel(orig_channel: str, video_name: str, candidates: list[str]) -> str:
    """hash 기반으로 결정론적 swap 채널 선택. 자기 자신 제외."""
    h = hashlib.sha256(f"{orig_channel}||{video_name}".encode()).hexdigest()
    seed = int(h[:8], 16)
    rng = random.Random(seed)
    pool = [c for c in candidates if c != orig_channel]
    return rng.choice(pool)


# 테스트
_test_ch = all_channels[0]
_test_vn = "test_video"
_test_swap = get_swap_channel(_test_ch, _test_vn, all_channels)
print(f"\nswap 테스트: {_test_ch!r} → {_test_swap!r}")

print(f"\n✅ 다음 셀에서 사용할 변수: all_channels, stats_df, get_swap_channel")

/home/taeyoung/miniconda3/envs/annotation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
eval.jsonl 파싱: 100%|██████████| 3320/3320 [00:00<00:00, 63840.54it/s]



eval.jsonl의 고유 채널 수: 664


채널 통계 계산: 100%|██████████| 664/664 [00:21<00:00, 30.63it/s]


✅ 전체 채널: 664개
        count       min        max      mean
bucket                                      
0         133  0.011472   0.664083  0.294273
1         133  0.680476   1.685595  1.158782
2         132  1.686024   3.237476  2.516964
3         133  3.251788   4.644189  3.934458
4         133  4.646865  16.830567  6.211632

swap 테스트: '\x08고기남자' → 'MBN MUSIC'

✅ 다음 셀에서 사용할 변수: all_channels, stats_df, get_swap_channel


In [2]:
# %% 셀 2: 기존 sglang 서버 종료 + ep3 merged 모델 서버 시작
import os
import subprocess
import time
import requests
import signal

os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

MODEL_PATH      = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"
PORT            = 8000
SERVER_LOG      = "sglang_server.log"
CONTEXT_LENGTH  = 131072


# -------------------------
# 1) 기존 서버 프로세스 종료
# -------------------------
print("🛑 기존 sglang 서버 종료 시도...")

if "server_process" in globals() and server_process.poll() is None:
    try:
        server_process.terminate()
        server_process.wait(timeout=10)
        print("   ✅ 이전 server_process 핸들로 종료")
    except Exception as e:
        try:
            server_process.kill()
            print(f"   ⚠️ terminate 실패 → kill ({e})")
        except Exception:
            pass

try:
    out = subprocess.run(
        ["lsof", "-t", f"-i:{PORT}"],
        capture_output=True, text=True, timeout=5,
    ).stdout.strip()
    pids = [int(p) for p in out.splitlines() if p.strip().isdigit()]
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
            print(f"   ✅ PID {pid} SIGTERM")
        except ProcessLookupError:
            pass
    time.sleep(3)
    out2 = subprocess.run(
        ["lsof", "-t", f"-i:{PORT}"],
        capture_output=True, text=True, timeout=5,
    ).stdout.strip()
    for p in out2.splitlines():
        if p.strip().isdigit():
            try:
                os.kill(int(p), signal.SIGKILL)
                print(f"   ✅ PID {p} SIGKILL")
            except ProcessLookupError:
                pass
except FileNotFoundError:
    print("   ⚠️ lsof 없음 — 포트 점유 확인 skip")

for _ in range(10):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=1)
        if r.status_code == 200:
            time.sleep(1)
            continue
    except Exception:
        break
print("   ✅ 포트 정리 완료\n")


# -------------------------
# 2) 서버 시작
# -------------------------
assert os.path.isdir(MODEL_PATH), f"모델 경로 없음: {MODEL_PATH}"

sglang_log = open(SERVER_LOG, "w")

server_process = subprocess.Popen(
    [
        "python", "-m", "sglang.launch_server",
        "--model-path", MODEL_PATH,
        "--port", str(PORT),
        "--mem-fraction-static", "0.8",
        "--context-length", str(CONTEXT_LENGTH),
        "--reasoning-parser", "qwen3",
        "--attention-backend", "triton",
    ],
    stdout=sglang_log,
    stderr=subprocess.STDOUT,
)


print(f"⏳ SGLang 서버 시작 중...")
print(f"   모델: {MODEL_PATH}")
print(f"   context-length: {CONTEXT_LENGTH}")

ready = False
for i in range(600):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=1)
        if r.status_code == 200:
            print(f"✅ 서버 준비 완료! ({i}초)")
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print(f"❌ 서버 시작 실패. 로그 확인: tail -n 200 {SERVER_LOG}")

🛑 기존 sglang 서버 종료 시도...
   ✅ 포트 정리 완료

⏳ SGLang 서버 시작 중...
   모델: ./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged
   context-length: 131072
✅ 서버 준비 완료! (24초)


In [4]:
# %% 셀 3: 664채널 × 5영상 × (본채널 + 랜덤1채널) inference
import os
import re
import json
import time
import httpx
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

EVAL_JSONL  = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR  = "./data/7_qwen_test"
MODEL_NAME  = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"

TEMPERATURE = 0.1
TOP_P       = 1.0
MAX_TOKENS  = 126976
PRESENCE_PENALTY = 1.5
FREQUENCY_PENALTY = 0.1

MAX_WORKERS = 20
TIMEOUT     = 3600

os.makedirs(OUTPUT_DIR, exist_ok=True)

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=httpx.Timeout(TIMEOUT, connect=5.0),
)


# -------------------------
# 1) eval.jsonl 전체 파싱
# -------------------------
def parse_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(
            part.get("text", "") for part in msg_content if isinstance(part, dict)
        )
    return ""


with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    eval_lines = f.readlines()

target_entries = []
for line in tqdm(eval_lines, desc="eval.jsonl 파싱"):
    ex = json.loads(line)
    msgs = ex["messages"]
    sys_msg  = next(m for m in msgs if m["role"] == "system")
    user_msg = next(m for m in msgs if m["role"] == "user")
    user_text = parse_user_text(user_msg["content"])

    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue

    channel = m_ch.group(1).strip()
    video   = m_vn.group(1).strip()

    target_entries.append({
        "channel":    channel,
        "video_name": video,
        "system":     sys_msg["content"],
        "user_text":  user_text,
    })

print(f"\n✅ 대상 영상 수: {len(target_entries)}")


# -------------------------
# 2) job 리스트 구성: 영상당 2건 (본채널 + 랜덤1채널)
# -------------------------
def swap_channel_line(user_text: str, new_channel: str) -> str:
    return re.sub(
        r"채널:\s*[^\n]+",
        f"채널: {new_channel}",
        user_text,
        count=1,
    )


def output_path(orig_ch: str, video: str, cond_ch: str) -> str:
    return os.path.join(OUTPUT_DIR, orig_ch, video, f"{cond_ch}.json")


jobs = []
for ent in tqdm(target_entries, desc="job 리스트 구성"):
    orig_ch = ent["channel"]
    video   = ent["video_name"]

    # (a) 본채널 조건
    out_a = output_path(orig_ch, video, orig_ch)
    if not os.path.exists(out_a):
        jobs.append({
            "orig_channel": orig_ch,
            "video_name":   video,
            "cond_channel": orig_ch,
            "system":       ent["system"],
            "user_text":    ent["user_text"],  # 원본 그대로
            "out_path":     out_a,
        })
    
    # (b) 랜덤 swap 채널 조건
    swap_ch = get_swap_channel(orig_ch, video, all_channels)
    out_b = output_path(orig_ch, video, swap_ch)
    if not os.path.exists(out_b):
        jobs.append({
            "orig_channel": orig_ch,
            "video_name":   video,
            "cond_channel": swap_ch,
            "system":       ent["system"],
            "user_text":    swap_channel_line(ent["user_text"], swap_ch),
            "out_path":     out_b,
        })
    

total_expected = len(target_entries) * 2
print(f"\n📋 실행할 job: {len(jobs)} (skip: {total_expected - len(jobs)})")

# -------------------------
# 3) 단일 요청 처리
# -------------------------
def call_model(job: dict) -> dict:
    t0 = time.time()
    try:
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": job["system"]},
                {"role": "user",   "content": job["user_text"]},
            ],
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        elapsed = time.time() - t0

        raw = resp.choices[0].message.content.strip()
        cleaned = raw
        if cleaned.startswith("```"):
            cleaned = re.sub(r"^```[a-zA-Z]*\n?", "", cleaned)
            cleaned = re.sub(r"\n?```$", "", cleaned)

        parsed = json.loads(cleaned)

        os.makedirs(os.path.dirname(job["out_path"]), exist_ok=True)
        record = {
            "orig_channel":      job["orig_channel"],
            "video_name":        job["video_name"],
            "cond_channel":      job["cond_channel"],
            "model":             MODEL_NAME,
            "params": {
                "temperature":   TEMPERATURE,
                "top_p":         TOP_P,
                "max_tokens":    MAX_TOKENS,
            },
            "system":            job["system"],
            "user_text":         job["user_text"],
            "raw_output":        raw,
            "instances":         parsed.get("instances", []) if isinstance(parsed, dict) else parsed,
            "elapsed_sec":       round(elapsed, 2),
            "prompt_tokens":     getattr(resp.usage, "prompt_tokens", None),
            "completion_tokens": getattr(resp.usage, "completion_tokens", None),
        }
        with open(job["out_path"], "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        return {"success": True, "elapsed": elapsed}

    except json.JSONDecodeError as e:
        return {"success": False, "error": f"JSONDecodeError: {str(e)[:100]}",
                "elapsed": time.time() - t0}
    except Exception as e:
        return {"success": False, "error": str(e)[:200],
                "elapsed": time.time() - t0}


# -------------------------
# 4) 병렬 실행
# -------------------------
def classify_error(err_msg: str) -> str:
    """에러 메시지를 카테고리로 분류"""
    if err_msg.startswith("JSONDecodeError"):
        return "json"
    low = err_msg.lower()
    if "readtimeout" in low or "read timeout" in low or "timed out" in low:
        return "timeout"
    if "connecterror" in low or "connection" in low or "remoteprotocol" in low:
        return "conn"
    if "contextlength" in low or "context length" in low or "maximum context" in low or "token" in low and "exceed" in low:
        return "ctxlen"
    if "500" in err_msg or "internal server" in low or "cuda" in low or "out of memory" in low or "oom" in low:
        return "server"
    if "400" in err_msg or "badrequest" in low or "bad request" in low:
        return "badreq"
    if "permission" in low or "no such file" in low or "disk" in low or "oserror" in low or "filenotfound" in low:
        return "io"
    if "indexerror" in low or "attributeerror" in low or "keyerror" in low or "typeerror" in low:
        return "resp"
    return "other"


if not jobs:
    print("\n✅ 모든 job이 이미 완료 — 새로 생성할 것 없음")
else:
    n_success = 0
    fail_counts = {
        "json":    0,  # JSONDecodeError (파싱 실패, max_tokens 도달 등)
        "timeout": 0,  # httpx ReadTimeout
        "conn":    0,  # 연결 끊김 / ConnectError
        "ctxlen":  0,  # context length 초과
        "server":  0,  # 서버 500 / CUDA OOM
        "badreq":  0,  # 400 Bad Request
        "io":      0,  # 파일 I/O 에러
        "resp":    0,  # resp 구조 이상 (IndexError 등)
        "other":   0,  # 기타
    }
    errors    = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(call_model, j): j for j in jobs}

        with tqdm(total=len(jobs), desc="inference") as pbar:
            for fut in as_completed(futures):
                job = futures[fut]
                res = fut.result()
                if res["success"]:
                    n_success += 1
                else:
                    cat = classify_error(res["error"])
                    fail_counts[cat] += 1
                    errors.append({
                        "orig":  job["orig_channel"],
                        "video": job["video_name"],
                        "cond":  job["cond_channel"],
                        "cat":   cat,
                        "err":   res["error"],
                    })
                pbar.update(1)
                pbar.set_postfix(
                    ok=n_success,
                    **{k: v for k, v in fail_counts.items() if v > 0},
                )

    n_fail = sum(fail_counts.values())
    print(f"\n📊 완료")
    print(f"  성공: {n_success}")
    print(f"  실패: {n_fail}")
    if n_fail > 0:
        print(f"\n  실패 원인별:")
        labels = {
            "json":    "JSON 파싱 실패",
            "timeout": "Read timeout",
            "conn":    "연결 에러",
            "ctxlen":  "Context length 초과",
            "server":  "서버 500 / OOM",
            "badreq":  "400 Bad request",
            "io":      "파일 I/O 에러",
            "resp":    "응답 구조 이상",
            "other":   "기타",
        }
        for k, v in fail_counts.items():
            if v > 0:
                print(f"    {labels[k]:20s}: {v}")
    if errors:
        print(f"\n실패 샘플 (최대 10개):")
        for e in errors[:10]:
            print(f"  [{e['cat']}][{e['orig']}] {e['video']} ← {e['cond']}: {e['err']}")

eval.jsonl 파싱: 100%|██████████| 3320/3320 [00:00<00:00, 59735.87it/s]



✅ 대상 영상 수: 3320


job 리스트 구성: 100%|██████████| 3320/3320 [00:00<00:00, 44159.39it/s]



📋 실행할 job: 1 (skip: 6639)


inference: 100%|██████████| 1/1 [00:11<00:00, 11.17s/it, ok=1]


📊 완료
  성공: 1
  실패: 0


In [10]:
# %% 메트릭 1: 인스턴스 개수 비율 |pred| / |GT|
import os, json, glob, re
import numpy as np
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"

# -------------------------
# 1) Pred 수집 (본채널만)
# -------------------------
pred_map = {}  # (ch, video) → n_instances
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True), desc="Pred 수집"):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    pred_map[key] = len(rec.get("instances", []))

# -------------------------
# 2) GT 수집 + 매칭
# -------------------------
pairs = []
with open(EVAL_JSONL, "r") as f:
    for line in tqdm(f, desc="GT 매칭"):
        ex = json.loads(line)
        msgs = ex["messages"]
        user_msg = next(m for m in msgs if m["role"] == "user")
        asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst_msg is None:
            continue

        user_text = user_msg["content"] if isinstance(user_msg["content"], str) else ""
        m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
        m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
        if not m_ch or not m_vn:
            continue

        key = (m_ch.group(1).strip(), m_vn.group(1).strip())
        if key not in pred_map:
            continue

        try:
            gt_inst = len(json.loads(asst_msg["content"]).get("instances", []))
        except:
            gt_inst = 0

        pairs.append({
            "channel": key[0],
            "video": key[1],
            "gt": gt_inst,
            "pred": pred_map[key],
        })

print(f"\n✅ 매칭된 페어: {len(pairs):,}개")

gt = np.array([p["gt"] for p in pairs])
pred = np.array([p["pred"] for p in pairs])

# -------------------------
# 3) 전체 통계
# -------------------------
ratio = pred / np.maximum(gt, 1)  # GT=0이면 1로 나눠서 무한대 방지

print(f"\n📊 인스턴스 개수 통계")
print(f"  {'':>15} {'GT':>10} {'Pred':>10}")
print(f"  {'mean':>15} {gt.mean():>10.1f} {pred.mean():>10.1f}")
print(f"  {'median':>15} {np.median(gt):>10.1f} {np.median(pred):>10.1f}")
print(f"  {'std':>15} {gt.std():>10.1f} {pred.std():>10.1f}")
print(f"  {'min':>15} {gt.min():>10} {pred.min():>10}")
print(f"  {'max':>15} {gt.max():>10} {pred.max():>10}")
print(f"  {'sum':>15} {gt.sum():>10,} {pred.sum():>10,}")

print(f"\n📊 |Pred| / |GT| 비율")
print(f"  mean:   {ratio.mean():.3f}")
print(f"  median: {np.median(ratio):.3f}")
print(f"  std:    {ratio.std():.3f}")

print(f"\n  {'구간':<20} {'샘플수':>8} {'비율':>8}")
print(f"  {'─'*20} {'─'*8} {'─'*8}")
bins = [(0, 0.5), (0.5, 0.8), (0.8, 1.2), (1.2, 2.0), (2.0, 5.0), (5.0, float('inf'))]
labels = ["< 0.5 (과소)", "0.5~0.8", "0.8~1.2 (정확)", "1.2~2.0", "2.0~5.0", "≥ 5.0 (과대)"]
for (lo, hi), label in zip(bins, labels):
    mask = (ratio >= lo) & (ratio < hi)
    cnt = mask.sum()
    print(f"  {label:<20} {cnt:>8} {cnt/len(ratio)*100:>7.1f}%")

# -------------------------
# 4) GT=0 / GT>0 분리
# -------------------------
gt_zero = gt == 0
gt_nonzero = gt > 0

print(f"\n📊 GT=0 (텔롭 없음): {gt_zero.sum()}개")
print(f"  Pred=0: {((gt_zero) & (pred == 0)).sum()} ({((gt_zero) & (pred == 0)).sum()/max(gt_zero.sum(),1)*100:.1f}%)")
print(f"  Pred>0: {((gt_zero) & (pred > 0)).sum()} ({((gt_zero) & (pred > 0)).sum()/max(gt_zero.sum(),1)*100:.1f}%) ← 환각")
if ((gt_zero) & (pred > 0)).sum() > 0:
    halluc_pred = pred[(gt_zero) & (pred > 0)]
    print(f"    환각 Pred 인스턴스 수 — mean: {halluc_pred.mean():.1f}, median: {np.median(halluc_pred):.1f}, max: {halluc_pred.max()}")

print(f"\n📊 GT>0 (텔롭 있음): {gt_nonzero.sum()}개")
ratio_nz = pred[gt_nonzero] / gt[gt_nonzero]
print(f"  |Pred|/|GT| — mean: {ratio_nz.mean():.3f}, median: {np.median(ratio_nz):.3f}")
print(f"  Pred=0 (누락): {(pred[gt_nonzero] == 0).sum()} ({(pred[gt_nonzero] == 0).sum()/gt_nonzero.sum()*100:.1f}%)")

# -------------------------
# 5) GT vs Pred 인스턴스 수 분포 비교
# -------------------------
print(f"\n📊 인스턴스 수 분포 비교 (GT vs Pred)")
print(f"  {'구간':<15} {'GT 샘플수':>10} {'Pred 샘플수':>12} {'GT %':>8} {'Pred %':>8}")
print(f"  {'─'*15} {'─'*10} {'─'*12} {'─'*8} {'─'*8}")

inst_bins = [0, 1, 10, 100, 1000, 10000, 100000]
for i in range(len(inst_bins) - 1):
    lo, hi = inst_bins[i], inst_bins[i+1]
    g_cnt = int(((gt >= lo) & (gt < hi)).sum())
    p_cnt = int(((pred >= lo) & (pred < hi)).sum())
    g_pct = g_cnt / len(gt) * 100
    p_pct = p_cnt / len(pred) * 100
    print(f"  {lo:>5}~{hi:<8} {g_cnt:>10,} {p_cnt:>12,} {g_pct:>7.1f}% {p_pct:>7.1f}%")

Pred 수집: 100%|██████████| 6630/6630 [00:01<00:00, 5741.41it/s]
GT 매칭: 3320it [00:00, 16536.93it/s]


✅ 매칭된 페어: 3,320개

📊 인스턴스 개수 통계
                          GT       Pred
             mean       57.2       58.6
           median       33.0       24.0
              std       90.6       97.3
              min          0          0
              max       1887       1000
              sum    189,874    194,589

📊 |Pred| / |GT| 비율
  mean:   4.563
  median: 0.600
  std:    17.582

  구간                        샘플수       비율
  ──────────────────── ──────── ────────
  < 0.5 (과소)               1413    42.6%
  0.5~0.8                   565    17.0%
  0.8~1.2 (정확)              345    10.4%
  1.2~2.0                   194     5.8%
  2.0~5.0                   432    13.0%
  ≥ 5.0 (과대)                371    11.2%

📊 GT=0 (텔롭 없음): 311개
  Pred=0: 156 (50.2%)
  Pred>0: 155 (49.8%) ← 환각
    환각 Pred 인스턴스 수 — mean: 23.9, median: 2.0, max: 183

📊 GT>0 (텔롭 있음): 3009개
  |Pred|/|GT| — mean: 3.805, median: 0.600
  Pred=0 (누락): 148 (4.9%)

📊 인스턴스 수 분포 비교 (GT vs Pred)
  구간                  GT 샘플수     Pred 샘플수  

In [5]:
# %% 메트릭 2: 타이밍 커버리지 (0.1초 bin 기반 binary F1 + count MAE)
import os, json, glob, re
import numpy as np
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"
BIN_SIZE = 0.1  # 0.1초


def instances_to_bins(instances, bin_size=BIN_SIZE):
    """인스턴스 리스트 → {bin_index: count} 딕셔너리"""
    bins = {}
    for inst in instances:
        s = inst["start_sec"]
        e = inst["end_sec"]
        b_start = int(s / bin_size)
        b_end = int(e / bin_size)
        for b in range(b_start, b_end):
            bins[b] = bins.get(b, 0) + 1
    return bins


def compute_coverage_metrics(gt_bins, pred_bins):
    """두 bin 딕셔너리에서 binary F1, count MAE 계산"""
    all_bins = set(gt_bins.keys()) | set(pred_bins.keys())

    if not all_bins:
        # 둘 다 비어있으면 완벽 일치
        return {"f1": 1.0, "precision": 1.0, "recall": 1.0, "count_mae": 0.0, "n_bins": 0}

    # binary F1
    tp = fp = fn = tn = 0
    count_errors = []

    for b in all_bins:
        g = b in gt_bins
        p = b in pred_bins
        if g and p:
            tp += 1
        elif not g and p:
            fp += 1
        elif g and not p:
            fn += 1
        else:
            tn += 1

        # count MAE: 해당 bin의 동시 텔롭 개수 차이
        g_count = gt_bins.get(b, 0)
        p_count = pred_bins.get(b, 0)
        count_errors.append(abs(g_count - p_count))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    count_mae = np.mean(count_errors) if count_errors else 0.0

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "count_mae": count_mae,
        "n_bins": len(all_bins),
    }


# -------------------------
# 1) Pred 수집 (본채널만)
# -------------------------
pred_map = {}
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True), desc="Pred 수집"):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    pred_map[key] = rec.get("instances", [])

# -------------------------
# 2) GT 수집 + 매칭 + 메트릭 계산
# -------------------------
results = []
with open(EVAL_JSONL, "r") as f:
    for line in tqdm(f, desc="GT 매칭 + 메트릭 계산"):
        ex = json.loads(line)
        msgs = ex["messages"]
        user_msg = next(m for m in msgs if m["role"] == "user")
        asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst_msg is None:
            continue

        user_text = user_msg["content"] if isinstance(user_msg["content"], str) else ""
        m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
        m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
        if not m_ch or not m_vn:
            continue

        key = (m_ch.group(1).strip(), m_vn.group(1).strip())
        if key not in pred_map:
            continue

        try:
            gt_instances = json.loads(asst_msg["content"]).get("instances", [])
        except:
            gt_instances = []

        gt_bins = instances_to_bins(gt_instances)
        pred_bins = instances_to_bins(pred_map[key])
        metrics = compute_coverage_metrics(gt_bins, pred_bins)

        results.append({
            "channel": key[0],
            "video": key[1],
            "gt_n": len(gt_instances),
            "pred_n": len(pred_map[key]),
            **metrics,
        })

print(f"\n✅ 매칭된 페어: {len(results):,}개")

# -------------------------
# 3) 전체 통계
# -------------------------
f1s = np.array([r["f1"] for r in results])
precs = np.array([r["precision"] for r in results])
recs = np.array([r["recall"] for r in results])
maes = np.array([r["count_mae"] for r in results])

print(f"\n📊 타이밍 커버리지 (전체)")
print(f"  {'메트릭':<15} {'mean':>8} {'median':>8} {'std':>8} {'min':>8} {'max':>8}")
print(f"  {'─'*15} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for name, arr in [("Binary F1", f1s), ("Precision", precs), ("Recall", recs), ("Count MAE", maes)]:
    print(f"  {name:<15} {arr.mean():>8.3f} {np.median(arr):>8.3f} {arr.std():>8.3f} {arr.min():>8.3f} {arr.max():>8.3f}")

# -------------------------
# 4) F1 분포
# -------------------------
print(f"\n📊 Binary F1 분포")
print(f"  {'구간':<15} {'샘플수':>8} {'비율':>8}")
print(f"  {'─'*15} {'─'*8} {'─'*8}")
f1_bins = [(0, 0.1), (0.1, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 0.9), (0.9, 1.001)]
f1_labels = ["0~0.1", "0.1~0.2", "0.2~0.4", "0.4~0.6", "0.6~0.8", "0.8~0.9", "0.9~1.0"]
for (lo, hi), label in zip(f1_bins, f1_labels):
    mask = (f1s >= lo) & (f1s < hi)
    cnt = mask.sum()
    print(f"  {label:<15} {cnt:>8} {cnt/len(f1s)*100:>7.1f}%")

# -------------------------
# 5) GT 인스턴스 구간별 F1, Count MAE
# -------------------------
gt_ns = np.array([r["gt_n"] for r in results])

print(f"\n📊 GT 인스턴스 구간별 타이밍 커버리지")
print(f"  {'GT 구간':<15} {'n':>6} {'F1 mean':>10} {'F1 med':>10} {'MAE mean':>10}")
print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10} {'─'*10}")

inst_bins = [0, 1, 10, 100, 1000, 10000, 100000]
for i in range(len(inst_bins) - 1):
    lo, hi = inst_bins[i], inst_bins[i+1]
    mask = (gt_ns >= lo) & (gt_ns < hi)
    if mask.sum() == 0:
        continue
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {f1s[mask].mean():>10.3f} {np.median(f1s[mask]):>10.3f} {maes[mask].mean():>10.3f}")

# -------------------------
# 6) GT=0 vs GT>0 분리
# -------------------------
gt_zero = gt_ns == 0
gt_nonzero = gt_ns > 0

print(f"\n📊 GT=0: {gt_zero.sum()}개")
print(f"  F1 mean: {f1s[gt_zero].mean():.3f} (Pred도 빈 경우 F1=1.0)")
print(f"  F1=1.0: {(f1s[gt_zero] == 1.0).sum()} ({(f1s[gt_zero] == 1.0).sum()/max(gt_zero.sum(),1)*100:.1f}%)")
print(f"  F1=0.0: {(f1s[gt_zero] == 0.0).sum()} ({(f1s[gt_zero] == 0.0).sum()/max(gt_zero.sum(),1)*100:.1f}%)")

print(f"\n📊 GT>0: {gt_nonzero.sum()}개")
print(f"  F1 mean: {f1s[gt_nonzero].mean():.3f}, median: {np.median(f1s[gt_nonzero]):.3f}")
print(f"  Count MAE mean: {maes[gt_nonzero].mean():.3f}, median: {np.median(maes[gt_nonzero]):.3f}")

Pred 수집: 100%|██████████| 6630/6630 [00:01<00:00, 6265.70it/s]
GT 매칭 + 메트릭 계산: 3320it [00:00, 4601.74it/s]


✅ 매칭된 페어: 3,320개

📊 타이밍 커버리지 (전체)
  메트릭                 mean   median      std      min      max
  ─────────────── ──────── ──────── ──────── ──────── ────────
  Binary F1          0.744    0.988    0.388    0.000    1.000
  Precision          0.761    1.000    0.392    0.000    1.000
  Recall             0.795    1.000    0.363    0.000    1.000
  Count MAE          1.806    1.039    8.651    0.000  348.000

📊 Binary F1 분포
  구간                   샘플수       비율
  ─────────────── ──────── ────────
  0~0.1                586    17.7%
  0.1~0.2               54     1.6%
  0.2~0.4              120     3.6%
  0.4~0.6              113     3.4%
  0.6~0.8              148     4.5%
  0.8~0.9              135     4.1%
  0.9~1.0             2164    65.2%

📊 GT 인스턴스 구간별 타이밍 커버리지
  GT 구간                n    F1 mean     F1 med   MAE mean
  ─────────────── ────── ────────── ────────── ──────────
      0~1           311      0.502      1.000      0.736
      1~10          717      0.394      0.075     

In [6]:
# %% 메트릭 3-1: 텍스트 유사도 (절대적 품질, GT vs pred_orig) - 32코어 병렬
import os, json, glob, re
import numpy as np
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from concurrent.futures import ProcessPoolExecutor, as_completed

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"
BIN_SIZE = 0.1


def instances_to_text_bins(instances, bin_size=BIN_SIZE):
    bins = {}
    for inst in instances:
        s = inst["start_sec"]
        e = inst["end_sec"]
        text = inst.get("text", "")
        b_start = int(s / bin_size)
        b_end = int(e / bin_size)
        for b in range(b_start, b_end):
            if b not in bins:
                bins[b] = set()
            bins[b].add(text)
    return bins


def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 1.0
    return SequenceMatcher(None, s1, s2).ratio()


def best_match_similarity(gt_texts, pred_texts):
    if not gt_texts or not pred_texts:
        return 0.0
    gt_list = list(gt_texts)
    pred_list = list(pred_texts)
    gt_to_pred = [max(normalized_levenshtein(g, p) for p in pred_list) for g in gt_list]
    pred_to_gt = [max(normalized_levenshtein(p, g) for g in gt_list) for p in pred_list]
    return (np.mean(gt_to_pred) + np.mean(pred_to_gt)) / 2


def compute_one(args):
    channel, video, gt_n, pred_n, gt_instances, pred_instances = args

    gt_bins = instances_to_text_bins(gt_instances)
    pred_bins = instances_to_text_bins(pred_instances)

    if not gt_bins and not pred_bins:
        return {"channel": channel, "video": video, "gt_n": gt_n, "pred_n": pred_n,
                "text_sim": 1.0, "n_overlap_bins": 0, "n_gt_bins": 0, "n_pred_bins": 0}

    overlap_bins = set(gt_bins.keys()) & set(pred_bins.keys())

    if not overlap_bins:
        return {"channel": channel, "video": video, "gt_n": gt_n, "pred_n": pred_n,
                "text_sim": 0.0, "n_overlap_bins": 0,
                "n_gt_bins": len(gt_bins), "n_pred_bins": len(pred_bins)}

    bin_sims = [best_match_similarity(gt_bins[b], pred_bins[b]) for b in overlap_bins]

    return {
        "channel": channel, "video": video, "gt_n": gt_n, "pred_n": pred_n,
        "text_sim": np.mean(bin_sims),
        "n_overlap_bins": len(overlap_bins),
        "n_gt_bins": len(gt_bins),
        "n_pred_bins": len(pred_bins),
    }


# -------------------------
# 1) Pred 수집 (본채널만)
# -------------------------
pred_map = {}
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True), desc="Pred 수집"):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    pred_map[key] = rec.get("instances", [])

# -------------------------
# 2) GT 수집 + task 구성
# -------------------------
tasks = []
with open(EVAL_JSONL, "r") as f:
    for line in f:
        ex = json.loads(line)
        msgs = ex["messages"]
        user_msg = next(m for m in msgs if m["role"] == "user")
        asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst_msg is None:
            continue

        user_text = user_msg["content"] if isinstance(user_msg["content"], str) else ""
        m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
        m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
        if not m_ch or not m_vn:
            continue

        key = (m_ch.group(1).strip(), m_vn.group(1).strip())
        if key not in pred_map:
            continue

        try:
            gt_instances = json.loads(asst_msg["content"]).get("instances", [])
        except:
            gt_instances = []

        tasks.append((key[0], key[1], len(gt_instances), len(pred_map[key]),
                       gt_instances, pred_map[key]))

print(f"🎯 {len(tasks):,}개 계산 예정")

# -------------------------
# 3) 병렬 실행
# -------------------------
results = []
with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(compute_one, t) for t in tasks]
    for f in tqdm(as_completed(futures), total=len(futures), desc="텍스트 유사도 계산"):
        results.append(f.result())

print(f"\n✅ 매칭된 페어: {len(results):,}개")

# -------------------------
# 4) 전체 통계
# -------------------------
sims = np.array([r["text_sim"] for r in results])
gt_ns = np.array([r["gt_n"] for r in results])

print(f"\n📊 텍스트 유사도 (전체)")
print(f"  mean:   {sims.mean():.3f}")
print(f"  median: {np.median(sims):.3f}")
print(f"  std:    {sims.std():.3f}")

# -------------------------
# 5) 유사도 분포
# -------------------------
print(f"\n📊 텍스트 유사도 분포")
print(f"  {'구간':<15} {'샘플수':>8} {'비율':>8}")
print(f"  {'─'*15} {'─'*8} {'─'*8}")
sim_bins = [(0, 0.1), (0.1, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 0.9), (0.9, 1.001)]
sim_labels = ["0~0.1", "0.1~0.2", "0.2~0.4", "0.4~0.6", "0.6~0.8", "0.8~0.9", "0.9~1.0"]
for (lo, hi), label in zip(sim_bins, sim_labels):
    mask = (sims >= lo) & (sims < hi)
    cnt = mask.sum()
    print(f"  {label:<15} {cnt:>8} {cnt/len(sims)*100:>7.1f}%")

# -------------------------
# 6) GT 인스턴스 구간별
# -------------------------
print(f"\n📊 GT 인스턴스 구간별 텍스트 유사도")
print(f"  {'GT 구간':<15} {'n':>6} {'sim mean':>10} {'sim med':>10}")
print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10}")

inst_bins = [0, 1, 10, 100, 1000, 10000]
for i in range(len(inst_bins) - 1):
    lo, hi = inst_bins[i], inst_bins[i+1]
    mask = (gt_ns >= lo) & (gt_ns < hi)
    if mask.sum() == 0:
        continue
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {sims[mask].mean():>10.3f} {np.median(sims[mask]):>10.3f}")

# -------------------------
# 7) GT=0 vs GT>0
# -------------------------
gt_zero = gt_ns == 0
gt_nonzero = gt_ns > 0

print(f"\n📊 GT=0: {gt_zero.sum()}개 → sim mean: {sims[gt_zero].mean():.3f}")
print(f"📊 GT>0: {gt_nonzero.sum()}개 → sim mean: {sims[gt_nonzero].mean():.3f}, median: {np.median(sims[gt_nonzero]):.3f}")

Pred 수집: 100%|██████████| 6630/6630 [00:01<00:00, 6176.52it/s]


🎯 3,320개 계산 예정


텍스트 유사도 계산: 100%|██████████| 3320/3320 [00:07<00:00, 454.08it/s]


✅ 매칭된 페어: 3,320개

📊 텍스트 유사도 (전체)
  mean:   0.409
  median: 0.404
  std:    0.306

📊 텍스트 유사도 분포
  구간                   샘플수       비율
  ─────────────── ──────── ────────
  0~0.1                747    22.5%
  0.1~0.2              231     7.0%
  0.2~0.4              666    20.1%
  0.4~0.6              766    23.1%
  0.6~0.8              514    15.5%
  0.8~0.9              121     3.6%
  0.9~1.0              275     8.3%

📊 GT 인스턴스 구간별 텍스트 유사도
  GT 구간                n   sim mean    sim med
  ─────────────── ────── ────────── ──────────
      0~1           311      0.502      1.000
      1~10          717      0.255      0.000
     10~100        1724      0.453      0.463
    100~1000        564      0.417      0.411
   1000~10000         4      0.300      0.271

📊 GT=0: 311개 → sim mean: 0.502
📊 GT>0: 3009개 → sim mean: 0.399, median: 0.404


In [7]:
# %% 메트릭: 상대적 차이 (pred_orig vs pred_swap, Wilcoxon signed-rank test)
import os, json, glob, re
import numpy as np
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.stats import wilcoxon

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"
BIN_SIZE = 0.1


# -------------------------
# 공통 함수
# -------------------------
def instances_to_bins(instances, bin_size=BIN_SIZE):
    bins = {}
    for inst in instances:
        b_start = int(inst["start_sec"] / bin_size)
        b_end = int(inst["end_sec"] / bin_size)
        for b in range(b_start, b_end):
            bins[b] = bins.get(b, 0) + 1
    return bins


def instances_to_text_bins(instances, bin_size=BIN_SIZE):
    bins = {}
    for inst in instances:
        b_start = int(inst["start_sec"] / bin_size)
        b_end = int(inst["end_sec"] / bin_size)
        for b in range(b_start, b_end):
            if b not in bins:
                bins[b] = set()
            bins[b].add(inst.get("text", ""))
    return bins


def compute_f1(gt_bins, pred_bins):
    all_bins = set(gt_bins.keys()) | set(pred_bins.keys())
    if not all_bins:
        return 1.0
    tp = sum(1 for b in all_bins if b in gt_bins and b in pred_bins)
    fp = sum(1 for b in all_bins if b not in gt_bins and b in pred_bins)
    fn = sum(1 for b in all_bins if b in gt_bins and b not in pred_bins)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0


def compute_count_mae(gt_bins, pred_bins):
    all_bins = set(gt_bins.keys()) | set(pred_bins.keys())
    if not all_bins:
        return 0.0
    return np.mean([abs(gt_bins.get(b, 0) - pred_bins.get(b, 0)) for b in all_bins])


def best_match_similarity(gt_texts, pred_texts):
    if not gt_texts or not pred_texts:
        return 0.0
    gt_list = list(gt_texts)
    pred_list = list(pred_texts)
    gt_to_pred = [max(SequenceMatcher(None, g, p).ratio() for p in pred_list) for g in gt_list]
    pred_to_gt = [max(SequenceMatcher(None, p, g).ratio() for g in gt_list) for p in pred_list]
    return (np.mean(gt_to_pred) + np.mean(pred_to_gt)) / 2


def compute_text_sim(gt_instances, pred_instances):
    gt_bins = instances_to_text_bins(gt_instances)
    pred_bins = instances_to_text_bins(pred_instances)
    if not gt_bins and not pred_bins:
        return 1.0
    overlap = set(gt_bins.keys()) & set(pred_bins.keys())
    if not overlap:
        return 0.0
    return np.mean([best_match_similarity(gt_bins[b], pred_bins[b]) for b in overlap])


def compute_all_metrics(gt_instances, pred_instances):
    gt_n = len(gt_instances)
    pred_n = len(pred_instances)

    # 인스턴스 개수 비율
    inst_ratio = pred_n / max(gt_n, 1)

    # 타이밍 커버리지
    gt_bins = instances_to_bins(gt_instances)
    pred_bins = instances_to_bins(pred_instances)
    f1 = compute_f1(gt_bins, pred_bins)
    count_mae = compute_count_mae(gt_bins, pred_bins)

    # 텍스트 유사도
    text_sim = compute_text_sim(gt_instances, pred_instances)

    return {
        "gt_n": gt_n, "pred_n": pred_n,
        "inst_ratio": inst_ratio,
        "f1": f1, "count_mae": count_mae,
        "text_sim": text_sim,
    }


def compute_one(args):
    channel, video, gt_instances, orig_instances, swap_instances = args
    orig_metrics = compute_all_metrics(gt_instances, orig_instances)
    swap_metrics = compute_all_metrics(gt_instances, swap_instances)
    return {
        "channel": channel, "video": video,
        "gt_n": orig_metrics["gt_n"],
        "orig_inst_ratio": orig_metrics["inst_ratio"],
        "swap_inst_ratio": swap_metrics["inst_ratio"],
        "orig_f1": orig_metrics["f1"],
        "swap_f1": swap_metrics["f1"],
        "orig_mae": orig_metrics["count_mae"],
        "swap_mae": swap_metrics["count_mae"],
        "orig_text_sim": orig_metrics["text_sim"],
        "swap_text_sim": swap_metrics["text_sim"],
    }


# -------------------------
# 1) Pred 수집 (orig + swap)
# -------------------------
orig_map = {}  # (ch, video) → instances
swap_map = {}  # (ch, video) → instances

for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True), desc="Pred 수집"):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    key = (rec["orig_channel"], rec["video_name"])
    instances = rec.get("instances", [])
    if rec.get("orig_channel") == rec.get("cond_channel"):
        orig_map[key] = instances
    else:
        swap_map[key] = instances

print(f"  orig: {len(orig_map):,}개, swap: {len(swap_map):,}개")

# -------------------------
# 2) GT + task 구성 (orig & swap 둘 다 있는 것만)
# -------------------------
tasks = []
with open(EVAL_JSONL, "r") as f:
    for line in f:
        ex = json.loads(line)
        msgs = ex["messages"]
        user_msg = next(m for m in msgs if m["role"] == "user")
        asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst_msg is None:
            continue

        user_text = user_msg["content"] if isinstance(user_msg["content"], str) else ""
        m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
        m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
        if not m_ch or not m_vn:
            continue

        key = (m_ch.group(1).strip(), m_vn.group(1).strip())
        if key not in orig_map or key not in swap_map:
            continue

        try:
            gt_instances = json.loads(asst_msg["content"]).get("instances", [])
        except:
            gt_instances = []

        tasks.append((key[0], key[1], gt_instances, orig_map[key], swap_map[key]))

print(f"🎯 orig & swap 둘 다 있는 페어: {len(tasks):,}개")

# -------------------------
# 3) 병렬 실행
# -------------------------
results = []
with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(compute_one, t) for t in tasks]
    for f in tqdm(as_completed(futures), total=len(futures), desc="메트릭 계산"):
        results.append(f.result())

print(f"\n✅ 완료: {len(results):,}개")

# -------------------------
# 4) 델타 계산 + Wilcoxon signed-rank test
# -------------------------
delta_f1 = np.array([r["orig_f1"] - r["swap_f1"] for r in results])
delta_mae = np.array([r["swap_mae"] - r["orig_mae"] for r in results])  # MAE는 낮을수록 좋으니 반대
delta_text = np.array([r["orig_text_sim"] - r["swap_text_sim"] for r in results])

print(f"\n📊 상대적 차이 (Δ = orig - swap, 양수면 orig가 나음)")
print(f"  {'메트릭':<20} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8} {'Wilcoxon p':>12}")
print(f"  {'─'*20} {'─'*10} {'─'*10} {'─'*8} {'─'*12}")

for name, delta in [("Binary F1", delta_f1), ("Count MAE (반전)", delta_mae), ("Text Similarity", delta_text)]:
    pos_pct = (delta > 0).sum() / len(delta) * 100
    # Wilcoxon: 0인 값 제외
    nonzero = delta[delta != 0]
    if len(nonzero) > 10:
        stat, p_val = wilcoxon(nonzero)
        p_str = f"{p_val:.2e}"
    else:
        p_str = "N/A"
    print(f"  {name:<20} {delta.mean():>+10.4f} {np.median(delta):>+10.4f} {pos_pct:>7.1f}% {p_str:>12}")

# -------------------------
# 5) 메트릭별 상세
# -------------------------
for name, orig_key, swap_key in [
    ("Binary F1", "orig_f1", "swap_f1"),
    ("Text Similarity", "orig_text_sim", "swap_text_sim"),
]:
    orig_arr = np.array([r[orig_key] for r in results])
    swap_arr = np.array([r[swap_key] for r in results])
    print(f"\n📊 {name} 상세")
    print(f"  {'':>15} {'orig':>10} {'swap':>10}")
    print(f"  {'mean':>15} {orig_arr.mean():>10.3f} {swap_arr.mean():>10.3f}")
    print(f"  {'median':>15} {np.median(orig_arr):>10.3f} {np.median(swap_arr):>10.3f}")

# -------------------------
# 6) GT 인스턴스 구간별 델타
# -------------------------
gt_ns = np.array([r["gt_n"] for r in results])

print(f"\n📊 GT 인스턴스 구간별 Δ F1 (orig - swap)")
print(f"  {'GT 구간':<15} {'n':>6} {'Δ mean':>10} {'Δ med':>10} {'Δ>0 %':>8}")
print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10} {'─'*8}")

inst_bins = [0, 1, 10, 100, 1000, 10000]
for i in range(len(inst_bins) - 1):
    lo, hi = inst_bins[i], inst_bins[i+1]
    mask = (gt_ns >= lo) & (gt_ns < hi)
    if mask.sum() == 0:
        continue
    d = delta_f1[mask]
    pos = (d > 0).sum() / len(d) * 100
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {pos:>7.1f}%")

print(f"\n📊 GT 인스턴스 구간별 Δ Text Sim (orig - swap)")
print(f"  {'GT 구간':<15} {'n':>6} {'Δ mean':>10} {'Δ med':>10} {'Δ>0 %':>8}")
print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10} {'─'*8}")

for i in range(len(inst_bins) - 1):
    lo, hi = inst_bins[i], inst_bins[i+1]
    mask = (gt_ns >= lo) & (gt_ns < hi)
    if mask.sum() == 0:
        continue
    d = delta_text[mask]
    pos = (d > 0).sum() / len(d) * 100
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {pos:>7.1f}%")

Pred 수집: 100%|██████████| 6630/6630 [00:01<00:00, 6255.80it/s]


  orig: 3,315개, swap: 3,315개
🎯 orig & swap 둘 다 있는 페어: 3,320개


메트릭 계산: 100%|██████████| 3320/3320 [00:13<00:00, 251.21it/s]


✅ 완료: 3,320개

📊 상대적 차이 (Δ = orig - swap, 양수면 orig가 나음)
  메트릭                      Δ mean   Δ median    Δ>0 %   Wilcoxon p
  ──────────────────── ────────── ────────── ──────── ────────────
  Binary F1               +0.0685    +0.0000    31.4%     2.54e-21
  Count MAE (반전)          +0.8994    +0.4838    66.3%    1.83e-165
  Text Similarity         +0.1869    +0.0910    66.3%    3.36e-260

📊 Binary F1 상세
                        orig       swap
             mean      0.744      0.675
           median      0.988      0.958

📊 Text Similarity 상세
                        orig       swap
             mean      0.409      0.222
           median      0.404      0.180

📊 GT 인스턴스 구간별 Δ F1 (orig - swap)
  GT 구간                n     Δ mean      Δ med    Δ>0 %
  ─────────────── ────── ────────── ────────── ────────
      0~1           311    +0.3891    +0.0000    42.1%
      1~10          717    +0.0396    +0.0000    26.2%
     10~100        1724    +0.0388    +0.0000    33.2%
    100~1000        

In [8]:
# %% start_sec 누락된 swap 파일 찾기
import os, json, glob
from tqdm.auto import tqdm

OUTPUT_DIR = "./data/7_qwen_test"

bad_files = []
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True), desc="검사"):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") == rec.get("cond_channel"):
        continue  # swap만 검사
    for i, inst in enumerate(rec.get("instances", [])):
        if "start_sec" not in inst or "end_sec" not in inst:
            bad_files.append({
                "path": os.path.relpath(path, OUTPUT_DIR),
                "orig": rec.get("orig_channel"),
                "video": rec.get("video_name"),
                "cond": rec.get("cond_channel"),
                "idx": i,
                "inst": inst,
                "total_inst": len(rec.get("instances", [])),
            })
            break  # 파일당 1개만

print(f"\n문제 파일: {len(bad_files)}개\n")
for b in bad_files[:10]:
    print(f"  {b['path']}")
    print(f"    orig={b['orig']}, cond={b['cond']}, video={b['video'][:50]}")
    print(f"    인스턴스 {b['idx']}번: {b['inst']}")
    print(f"    전체 인스턴스: {b['total_inst']}개\n")

검사: 100%|██████████| 6630/6630 [00:01<00:00, 6419.20it/s]


문제 파일: 0개

